# Data cleaning

This notebook combines the official DfT collision files for 2023-2025, checks their fields, and creates a clean collision-level table. The primary outcome follows the recorded categories: Fatal or Serious = KSI; Slight = non-KSI. DfT adjusted severity values are retained as a probability-weighted sensitivity measure, not converted into hard labels.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
YEARS = [2023, 2024, 2025]
source_files = [RAW_DIR / f"collision-{year}.csv" for year in YEARS]
missing_files = [path for path in source_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Download the selected annual files first: {missing_files}")

raw = pd.concat(
    [pd.read_csv(path, low_memory=False) for path in source_files],
    ignore_index=True,
)

print(f"Raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
print("Columns:")
print(", ".join(raw.columns))
print("\nColumns with the most missing values:")
display(raw.isna().sum().sort_values(ascending=False).head(15).to_frame("missing"))

Raw shape: 306,710 rows x 44 columns
Columns:
collision_index, collision_year, collision_ref_no, location_easting_osgr, location_northing_osgr, longitude, latitude, police_force, collision_severity, number_of_vehicles, number_of_casualties, date, day_of_week, time, local_authority_district, local_authority_ons_district, local_authority_highway, local_authority_highway_current, first_road_class, first_road_number, road_type, speed_limit, junction_detail_historic, junction_detail, junction_control, second_road_class, second_road_number, pedestrian_crossing_human_control_historic, pedestrian_crossing_physical_facilities_historic, pedestrian_crossing, light_conditions, weather_conditions, road_surface_conditions, special_conditions_at_site, carriageway_hazards_historic, carriageway_hazards, urban_or_rural_area, did_police_officer_attend_scene_of_accident, trunk_road_flag, lsoa_of_accident_location, enhanced_severity_collision, collision_injury_based, collision_adjusted_severity_serious, co

,missing
location_easting_osgr,14
location_northing_osgr,14
longitude,14
latitude,14
collision_index,0
carriageway_hazards_historic,0
second_road_number,0
pedestrian_crossing_human_control_historic,0
pedestrian_crossing_physical_facilities_historic,0
pedestrian_crossing,0


## Cleaning rules

The labels below come from DfT's 2025 data guide. Codes marked unknown, unallocated, or out of range stay visible as categories rather than causing rows to disappear. The harmonized `junction_detail` field follows DfT's historical-to-current conversion table. Only records without a valid Fatal/Serious/Slight outcome are excluded; missing coordinates remain for the non-spatial analysis.

In [2]:
import numpy as np

sys.path.insert(0, str(PROJECT_ROOT))
from src.utils import map_codes

selected_columns = [
    "collision_index",
    "collision_year",
    "collision_severity",
    "speed_limit",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "urban_or_rural_area",
    "road_type",
    "junction_detail",
    "day_of_week",
    "time",
    "number_of_vehicles",
    "latitude",
    "longitude",
    "collision_adjusted_severity_serious",
    "collision_adjusted_severity_slight",
]
missing_columns = sorted(set(selected_columns) - set(raw.columns))
if missing_columns:
    raise ValueError(f"Expected DfT columns are missing: {missing_columns}")

work = raw[selected_columns].copy()

code_labels = {
    "collision_severity": {1: "Fatal", 2: "Serious", 3: "Slight"},
    "speed_limit": {
        20: "20-30 mph",
        30: "20-30 mph",
        40: "40 mph",
        50: "50 mph",
        60: "60 mph",
        70: "70 mph",
        99: "Unknown / missing",
        -1: "Unknown / missing",
    },
    "light_conditions": {
        1: "Daylight",
        4: "Darkness, lights lit",
        5: "Darkness, no road lighting",
        6: "Darkness, no road lighting",
        7: "Darkness, lighting unknown",
        -1: "Unknown / missing",
    },
    "weather_conditions": {
        1: "Fine, no high winds",
        2: "Rain, no high winds",
        3: "Snow, no high winds",
        4: "Fine, high winds",
        5: "Rain, high winds",
        6: "Snow, high winds",
        7: "Fog or mist",
        8: "Other",
        9: "Unknown / missing",
        -1: "Unknown / missing",
    },
    "road_surface_conditions": {
        1: "Dry",
        2: "Wet or damp",
        3: "Snow",
        4: "Frost or ice",
        5: "Flood water",
        6: "Oil or diesel",
        7: "Mud",
        9: "Unknown / missing",
        -1: "Unknown / missing",
    },
    "urban_or_rural_area": {
        1: "Urban",
        2: "Rural",
        3: "Unallocated",
        -1: "Unknown / missing",
    },
    "road_type": {
        1: "Roundabout",
        2: "One-way or slip road",
        3: "Dual carriageway",
        6: "Single carriageway",
        7: "One-way or slip road",
        9: "Unknown / missing",
        12: "One-way or slip road",
        -1: "Unknown / missing",
    },
    "junction_detail": {
        0: "Not at or near a junction",
        13: "T or staggered junction",
        16: "Crossroads",
        17: "Junction with more than four arms",
        18: "Private drive or entrance",
        19: "Other junction",
        99: "Unknown / missing",
        -1: "Unknown / missing",
    },
    "day_of_week": {
        1: "Sunday",
        2: "Monday",
        3: "Tuesday",
        4: "Wednesday",
        5: "Thursday",
        6: "Friday",
        7: "Saturday",
        -1: "Unknown / missing",
    },
}

for column, labels in code_labels.items():
    mapped = map_codes(work[column], labels)
    work[column] = mapped.where(work[column].isin(labels), "Unknown / missing")

work["weather"] = work["weather_conditions"].replace(
    {
        "Fine, high winds": "Fine",
        "Fine, no high winds": "Fine",
        "Rain, high winds": "Rain",
        "Rain, no high winds": "Rain",
        "Snow, high winds": "Snow or fog",
        "Snow, no high winds": "Snow or fog",
        "Fog or mist": "Snow or fog",
    }
)
work["road_surface"] = work["road_surface_conditions"].replace(
    {
        "Snow": "Other adverse surface",
        "Frost or ice": "Other adverse surface",
        "Flood water": "Other adverse surface",
        "Oil or diesel": "Other adverse surface",
        "Mud": "Other adverse surface",
    }
)

hour = pd.to_numeric(
    work["time"].astype("string").str.extract(r"^(\d{1,2}):\d{2}$")[0],
    errors="coerce",
)
hour = hour.where(hour.between(0, 23))
work["hour"] = hour.astype("Int64")
time_bands = pd.cut(
    hour,
    bins=[-1, 5, 9, 15, 19, 23],
    labels=["Overnight", "Morning peak", "Daytime", "Evening peak", "Late evening"],
    include_lowest=True,
)
work["time_of_day"] = time_bands.astype("string").fillna("Unknown / missing")

vehicles = pd.to_numeric(work["number_of_vehicles"], errors="coerce")
work["vehicle_count_band"] = np.select(
    [vehicles.eq(1), vehicles.eq(2), vehicles.ge(3)],
    ["1", "2", "3+"],
    default="Unknown / missing",
)

work["ksi_adjusted_probability"] = pd.to_numeric(
    work["collision_adjusted_severity_serious"], errors="coerce"
)
work.loc[work["collision_severity"].eq("Fatal"), "ksi_adjusted_probability"] = 1.0

rows_before = len(work)
valid_severity = work["collision_severity"].isin(["Fatal", "Serious", "Slight"])
rows_without_valid_severity = int((~valid_severity).sum())
work = work.loc[valid_severity].copy()
work["ksi"] = work["collision_severity"].isin(["Fatal", "Serious"]).astype("int8")

clean = work.rename(
    columns={
        "collision_index": "collision_id",
        "collision_year": "year",
        "collision_severity": "severity",
        "speed_limit": "speed_limit_band",
        "light_conditions": "light",
        "urban_or_rural_area": "urban_rural",
        "day_of_week": "day_of_week",
    }
)[
    [
        "collision_id",
        "year",
        "severity",
        "ksi",
        "speed_limit_band",
        "light",
        "weather",
        "road_surface",
        "urban_rural",
        "road_type",
        "junction_detail",
        "day_of_week",
        "hour",
        "time_of_day",
        "number_of_vehicles",
        "vehicle_count_band",
        "latitude",
        "longitude",
        "ksi_adjusted_probability",
    ]
]

print(f"Rows before cleaning: {rows_before:,}")
print(f"Rows excluded for invalid severity: {rows_without_valid_severity:,}")
print(f"Rows after cleaning: {len(clean):,}")
print(f"Reported KSI collisions: {clean['ksi'].sum():,}")
print(f"Reported KSI rate: {clean['ksi'].mean():.2%}")
print(
    "Adjusted expected KSI rate (sensitivity only): "
    f"{clean['ksi_adjusted_probability'].mean():.2%}"
)
print(f"Cleaned table shape: {clean.shape}")

Rows before cleaning: 306,710
Rows excluded for invalid severity: 0
Rows after cleaning: 306,710
Reported KSI collisions: 76,673
Reported KSI rate: 25.00%
Adjusted expected KSI rate (sensitivity only): 25.94%
Cleaned table shape: (306710, 19)


## Save the cleaned file

The processed file keeps both the binary reported outcome and the separate DfT-adjusted probability. Rows with missing coordinates are retained here and can be filtered only for mapping.

In [3]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output_file = PROCESSED_DIR / "collisions_clean.csv"
clean.to_csv(output_file, index=False)

print(f"Saved {len(clean):,} rows to {output_file.relative_to(PROJECT_ROOT)}")
display(clean.head(5))

Saved 306,710 rows to data/processed/collisions_clean.csv


,collision_id,year,severity,ksi,speed_limit_band,light,weather,road_surface,urban_rural,road_type,junction_detail,day_of_week,hour,time_of_day,number_of_vehicles,vehicle_count_band,latitude,longitude,ksi_adjusted_probability
0,2023170M20483,2023,Slight,0,20-30 mph,Daylight,Fine,Dry,Urban,Single carriageway,Crossroads,Saturday,12,Daytime,1,1,54.565548,-1.210024,0.03426
1,2023170M10633,2023,Serious,1,20-30 mph,"Darkness, lights lit",Fine,Dry,Urban,Single carriageway,T or staggered junction,Sunday,20,Late evening,2,2,54.562855,-1.224443,1.00000
2,2023170S11103,2023,Slight,0,20-30 mph,Daylight,Fine,Dry,Urban,Single carriageway,T or staggered junction,Tuesday,9,Morning peak,1,1,54.561334,-1.329383,0.05072
3,2023111343936,2023,Serious,1,60 mph,Daylight,Rain,Other adverse surface,Rural,Roundabout,Not at or near a junction,Monday,9,Morning peak,1,1,54.549692,-1.587814,1.00000
4,2023070054936,2023,Slight,0,20-30 mph,Daylight,Fine,Other adverse surface,Rural,Single carriageway,T or staggered junction,Thursday,8,Morning peak,2,2,53.432995,-2.581866,0.00366


The police-force code is carried through only to define the selected spatial view. DfT code 1 denotes the Metropolitan Police area.

In [4]:
clean["police_force_code"] = raw.loc[clean.index, "police_force"].astype("int16")
clean.to_csv(output_file, index=False)
print(f"Updated spatial field in {output_file.relative_to(PROJECT_ROOT)}")

Updated spatial field in data/processed/collisions_clean.csv
